# ⏳ Notebook 3: Long Polling

Long polling is the easiest way to achieve **near real-time** updates while still using standard HTTP. The server holds the request open until new data is available!

## Learning Objectives

By the end of this notebook, you'll understand:
- How long polling differs from simple polling
- The latency trade-offs of long polling
- How to implement a long polling server and client
- When to choose long polling over other methods

## 🤔 What is Long Polling?

With **simple polling**, the server responds immediately (even if there's nothing new).

With **long polling**, the server **waits** until there's new data before responding!

```
Simple Polling:                    Long Polling:
                                   
Client ─► Server                   Client ─► Server
Client ◄─ "nothing"  (immediate)   Client    ...waiting...
                                   Client    ...waiting...
Client ─► Server                   Client    ...waiting...
Client ◄─ "nothing"  (immediate)   Client ◄─ "new data!"  (when available)
                                   
Client ─► Server                   Client ─► Server
Client ◄─ "new data!"              Client    ...waiting...
```

The key insight: **the server holds the connection open** until it has something to say!

## 📊 Comparing Latency

Let's understand why long polling has lower latency:

```
Simple Polling (2s interval):
────────────────────────────────────────────────────►
     │Poll     │Poll     │Poll     │Poll
     ▼         ▼         ▼         ▼
                    💬 Message arrives here
                              │
                              └─► User sees it HERE (up to 2s later!)

Long Polling:
────────────────────────────────────────────────────►
     │Request held open.....................│
                    💬 Message arrives here
                    │
                    └─► User sees it IMMEDIATELY!
```

## 🛠️ Let's Build It!

### Step 1: Start the Server

The next cell starts `servers/long_polling_server.py` for you and shuts it
down when the kernel exits. To watch its log live instead, start it yourself
in a second terminal first — the notebook leaves an already-listening port
alone:

```bash
cd 04-patterns/real-time-updates/servers
python long_polling_server.py     # 🚀 Starting Long Polling Server on port 5002
```

In [ ]:
# Start the long polling server this notebook talks to.
#
# `ensure_server` is idempotent: if you already started the server by hand
# in another terminal it is left alone, otherwise it is launched as a
# background process using this notebook's own interpreter (the lab .venv)
# and shut down when the kernel exits. This is what makes the notebook
# runnable on its own -- previously the next cell just died with a raw
# ConnectionError if you had not started the server first.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "servers"))
from lab_servers import ensure_server

print(ensure_server(5002))

import requests

health = requests.get("http://localhost:5002/health", timeout=5)
health.raise_for_status()
print("health:", health.json())


### Step 2: Create the Long Polling Client

In [ ]:
import requests
import time
from datetime import datetime
import threading

class LongPollingClient:
    """
    A long polling client that holds requests open until new data arrives.
    """
    
    def __init__(self, server_url: str, timeout: int = 35, max_wait: float | None = None):
        self.server_url = server_url
        # HTTP timeout. ALWAYS keep this LONGER than the server's hold time --
        # see `max_wait` below and the note under the first experiment.
        self.timeout = timeout
        # How long we ask the SERVER to hold the request. None = the server's
        # own default (30s). This is the knob a demo should turn.
        self.max_wait = max_wait
        self.last_timestamp = 0
        self.running = False
        self.session = requests.Session()  # Reuse connections!
    
    def long_poll_once(self):
        """
        Make a single long poll request.
        This blocks until the server responds!
        """
        try:
            start_time = time.time()
            
            params = {"since": self.last_timestamp}
            if self.max_wait is not None:
                params["max_wait"] = self.max_wait
            
            response = self.session.get(
                f"{self.server_url}/messages/poll",
                params=params,
                timeout=self.timeout
            )
            
            elapsed = time.time() - start_time
            
            if response.status_code == 200:
                data = response.json()
                messages = data.get("messages", [])
                timeout = data.get("timeout", False)
                
                if messages:
                    self.last_timestamp = max(msg["timestamp"] for msg in messages)
                
                return {
                    "messages": messages,
                    "timeout": timeout,
                    "server_waited": data.get("waited"),
                    "wait_time": elapsed
                }
                
        except requests.exceptions.Timeout:
            # The CLIENT gave up first. This is the failure mode you want to
            # avoid: the server is still holding a waiter for a request whose
            # reader has vanished.
            return {"messages": [], "timeout": True, "client_aborted": True,
                    "wait_time": self.timeout}
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed: {e}")
            return {"messages": [], "error": str(e)}
    
    def send_message(self, user: str, text: str):
        """
        Send a message to the chat.
        """
        try:
            response = self.session.post(
                f"{self.server_url}/messages",
                json={"user": user, "text": text},
                timeout=5
            )
            return response.status_code == 201
        except:
            return False

# Create client
client = LongPollingClient("http://localhost:5002")
print("✅ Long polling client created!")

## 🧪 Experiment: See Long Polling in Action!

Let's see how the server holds the request open until data arrives.

In [ ]:
# First: prove the server really HOLDS the request open.
#
# This is the whole difference between long polling and short polling. If the
# request came back instantly with an empty list, we would just be doing
# notebook 2 with extra steps -- so we assert on the elapsed time.
#
# Note WHICH timeout we shorten. We ask the SERVER to give up after 5s
# (`max_wait`) and leave the client's HTTP timeout at 35s. The other way
# round -- a client that aborts before the server answers -- leaves the
# server holding a waiter for a reader that is already gone.

SERVER_HOLD = 5

client.timeout = 35        # client patience: always > server hold time
client.max_wait = SERVER_HOLD
client.last_timestamp = time.time()  # ask only for messages from now on

print(f"🔄 Long polling an idle server (server will hold for {SERVER_HOLD}s)...\n")

start = time.time()
result = client.long_poll_once()
elapsed = time.time() - start

print(f"⏱️  Request took: {elapsed:.2f} seconds")
print(f"   Server says it waited: {result.get('server_waited'):.2f} seconds")

if result.get("timeout"):
    print("⏰ Server gave up and answered 'nothing new' -- politely, on its own terms.")
else:
    print(f"📬 Got {len(result['messages'])} message(s)!")

# A short poll would return in milliseconds. Anything under ~80% of the hold
# time means the server is NOT holding the connection open.
assert elapsed > SERVER_HOLD * 0.8, (
    f"the server answered after only {elapsed:.2f}s -- it is not holding the "
    f"connection open, so this is short polling, not long polling"
)
assert not result.get("client_aborted"), "the client timed out first -- raise client.timeout"
print("\n✅ Connection was genuinely held open for the full window.")

In [ ]:
# Now the payoff: a message arrives mid-poll and the held request returns
# straight away -- it does NOT sit out the rest of the window.
import threading
import time

SEND_AFTER = 3
client.max_wait = 20         # plenty of headroom; we expect to return early
client.timeout = 35

def send_message_after_delay(delay):
    """Send a message after a delay."""
    time.sleep(delay)
    print(f"\n📤 [{datetime.now().strftime('%H:%M:%S')}] Sending message...")
    client.send_message("Alice", "Hello from a delayed send!")

client.last_timestamp = time.time()

print(f"🔄 [{datetime.now().strftime('%H:%M:%S')}] Starting long poll (server hold: {client.max_wait}s)...")
print(f"   A message will be sent in {SEND_AFTER} seconds...\n")

sender = threading.Thread(target=send_message_after_delay, args=(SEND_AFTER,))
sender.start()

start = time.time()
result = client.long_poll_once()
elapsed = time.time() - start

print(f"\n📬 [{datetime.now().strftime('%H:%M:%S')}] Long poll returned!")
print(f"⏱️  Wait time: {elapsed:.2f} seconds")
print(f"⚡ Delivery lag after the send: {(elapsed - SEND_AFTER)*1000:.0f}ms")

for msg in result["messages"]:
    print(f"   └─ {msg['user']}: {msg['text']}")

sender.join()

# The two things that would make this lesson silently false:
assert result["messages"], "the held poll missed the message entirely (lost wakeup)"
assert elapsed < client.max_wait * 0.5, (
    f"the poll ran for {elapsed:.2f}s -- it sat out the window instead of "
    f"waking on the publish"
)
assert elapsed - SEND_AFTER < 0.5, (
    f"woke {elapsed - SEND_AFTER:.2f}s after the send; expected near-instant"
)
print("\n✅ Woke within milliseconds of the publish -- that is the whole point.")

## 📊 The Latency Problem with High-Frequency Updates

Long polling has a subtle issue: after receiving a message, the client must make a **new request**. If messages arrive in quick succession, this introduces latency.

```
Timeline (100ms network latency):

0ms    Client sends request ──────────────────►
100ms  Request arrives at server

150ms  💬 Message 1 arrives at server
150ms  Server responds immediately ◄──────────
250ms  Client receives Message 1

160ms  💬 Message 2 arrives at server (10ms after Message 1!)
       But client doesn't know yet...

250ms  Client makes new request ──────────────►
350ms  Request arrives at server
350ms  Server sends Message 2 ◄───────────────
450ms  Client receives Message 2

Total latency for Message 2: 290ms (450ms - 160ms)
```

This is **worse** than the 100ms network latency!

> ⚠️ That timeline assumes a 100ms RTT. We are on **loopback**, where a round
> trip costs microseconds, so the gap will look tiny here — the next cell
> measures what it actually is rather than repeating the number above. Watch
> the **round-trip count** instead: that is the part that does not shrink when
> the network gets fast, and it is what SSE removes in the next notebook.

In [ ]:
# Let's MEASURE the burst behaviour rather than assert it.
#
# The claim under test is "each message costs its own long-poll round trip".
# That is true only for messages spaced further apart than the reconnect gap;
# anything published while the client is between requests gets batched into
# the next response. We measure both regimes.

def burst_while_polling(n=3, gap=0.1):
    """Publish `n` messages, `gap` apart, while a long poll is in flight."""
    client.last_timestamp = time.time()
    client.max_wait = 5
    client.timeout = 35

    sent_at = {}

    def send_burst():
        time.sleep(1)  # let the first long poll get established
        for i in range(n):
            text = f"Message {i+1}"
            sent_at[text] = time.time()
            print(f"📤 [{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] Sending {text}")
            client.send_message("Burst", text)
            time.sleep(gap)

    sender = threading.Thread(target=send_burst)
    sender.start()

    latencies, round_trips = {}, 0
    deadline = time.time() + 30
    while len(latencies) < n and time.time() < deadline:
        result = client.long_poll_once()
        round_trips += 1
        now = time.time()
        for msg in result["messages"]:
            latencies[msg["text"]] = now - sent_at[msg["text"]]
            print(f"📬 [{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] Received: "
                  f"{msg['text']} (latency {latencies[msg['text']]*1000:.0f}ms, "
                  f"round trip #{round_trips})")
    sender.join()
    return latencies, round_trips

def backlog_then_poll(n=3):
    """Publish `n` messages with NO poll in flight, then poll once."""
    client.last_timestamp = time.time()
    for i in range(n):
        client.send_message("Backlog", f"Backlog {i+1}")
    result = client.long_poll_once()
    return len(result["messages"])

print("📊 Case A: messages published DURING a held poll (100ms apart)")
print("="*60)
lat, trips = burst_while_polling()
worst = max(lat.values())
print(f"\n   → {len(lat)} messages, {trips} long-poll round trip(s), "
      f"worst latency {worst*1000:.0f}ms")

print("\n📊 Case B: messages published while NO poll is in flight")
print("="*60)
batched = backlog_then_poll()
print(f"   → {batched} messages returned by a SINGLE request")

print("\n💡 What the numbers say:")
print(f"   Case A cost one round trip per message ({trips} for {len(lat)}), because")
print("   our reconnect on loopback takes ~3ms -- far less than the 100ms send gap,")
print("   so the client is always already waiting when the next message lands.")
print(f"   Case B batched all {batched} into one response: the `since` cursor returns")
print("   everything newer, so the reconnect gap is where messages pile up.")
print("\n   That gap is the real cost. On loopback it is sub-millisecond. Add a 100ms")
print("   RTT and every message after the first waits for it -- the ~290ms in the")
print("   diagram above. SSE removes the gap entirely by never closing the response.")

assert len(lat) == 3, f"only {len(lat)}/3 burst messages arrived"
assert trips == 3, (
    f"expected one round trip per message when they are spaced further apart "
    f"than the reconnect gap, got {trips}"
)
assert batched == 3, (
    f"a backlog published between polls must come back in ONE response, got "
    f"{batched} -- the `since` cursor is dropping or splitting messages"
)

## ✅ Advantages of Long Polling

1. **Near real-time** - Much faster than simple polling
2. **Still just HTTP** - Works with existing infrastructure
3. **No special client libraries** - Standard HTTP client works
4. **Firewall friendly** - Just HTTP requests
5. **Stateless-ish** - Server doesn't need persistent state

## ❌ Disadvantages

1. **Latency for burst messages** - Must reconnect after each response
2. **Connection overhead** - Many held connections use resources
3. **Timeout complexity** - Need to handle timeouts gracefully
4. **Load balancer issues** - Long requests might be terminated
5. **Monitoring challenges** - Requests look "slow" in metrics

In [ ]:
# Check server stats to see how many clients are waiting

def check_server_stats():
    try:
        response = requests.get("http://localhost:5002/stats")
        stats = response.json()
        
        print("📊 Server Statistics")
        print("="*40)
        print(f"   Waiting clients: {stats['waiting_clients']}")
        print(f"   Total messages:  {stats['total_messages']}")
    except:
        print("❌ Could not fetch server stats")

check_server_stats()

## 🎯 When to Use Long Polling

Long polling is great for:

| Use Case | Why Long Polling Works |
|----------|----------------------|
| Payment status | Need to know ASAP when payment completes |
| Job completion | Background task finished notification |
| Infrequent updates | Not many messages, but need them fast |
| Simple setups | When WebSocket is overkill |
| Legacy systems | When WebSocket isn't supported |

### Don't use it when:

- High-frequency updates (chat with lots of messages)
- Bi-directional communication needed
- Very low latency required
- Many concurrent users (resource overhead)

## 🔧 Implementation Tips

### 1. Set Appropriate Timeouts

The server must give up **before** the client does. If the client aborts
first, the server is left holding a waiter for a reader that is already gone,
and it only notices when its own timer fires.

```python
# Server timeout must be SHORTER than the client timeout
SERVER_TIMEOUT = 30  # server answers "nothing new" at 30s
CLIENT_TIMEOUT = 35  # client only gives up if the server is truly stuck

# Also configure your load balancer -- it has its own idea of "too slow":
# nginx: proxy_read_timeout 60s;   (> both of the above)
```

That is exactly the pairing the cells above use: `client.max_wait` sets the
server's hold time, `client.timeout` stays comfortably above it.

### 2. Handle Reconnection

```python
while True:
    try:
        result = long_poll()
        process_messages(result)
    except Exception as e:
        # Wait before retrying on error
        time.sleep(1)
```

### 3. Use HTTP Keep-Alive

```python
# Reuse the connection for subsequent polls
session = requests.Session()
```

In [ ]:
# Let's implement a robust long polling loop

def robust_long_polling_loop(duration=15, expected_messages=2):
    """
    A production-ready long polling loop with error handling.
    """
    print(f"🔄 Starting robust long polling loop for {duration}s...\n")

    client.last_timestamp = time.time()
    client.max_wait = 4     # server gives up after 4s and we re-poll
    client.timeout = 35     # client patience stays well above that

    start_time = time.time()
    poll_count = 0
    error_count = 0
    messages_received = 0

    while time.time() - start_time < duration:
        try:
            poll_count += 1
            result = client.long_poll_once()
            # Stamp AFTER the call: long_poll_once() blocks, so a timestamp
            # taken before it would label the message with the time we asked,
            # not the time it arrived.
            current_time = datetime.now().strftime('%H:%M:%S')

            if result.get("error"):
                error_count += 1
                print(f"[{current_time}] ❌ Error - waiting 1s before retry")
                time.sleep(1)  # Backoff on error
                continue

            if result["messages"]:
                for msg in result["messages"]:
                    messages_received += 1
                    print(f"[{current_time}] 📬 {msg['user']}: {msg['text']}")
            elif result.get("timeout"):
                print(f"[{current_time}] ⏰ Server said 'nothing new' - reconnecting...")

        except KeyboardInterrupt:
            print("\n🛑 Stopped by user")
            break

    print(f"\n📊 Summary:")
    print(f"   Polls made: {poll_count}")
    print(f"   Messages received: {messages_received}")
    print(f"   Errors: {error_count}")
    print(f"\n💡 {poll_count} requests for {messages_received} messages. Simple polling")
    print(f"   at 1s would have made ~{duration} requests for the same two events.")

    assert messages_received == expected_messages, (
        f"expected {expected_messages} messages, got {messages_received} -- the "
        f"loop is dropping updates across reconnects"
    )
    assert error_count == 0, f"{error_count} errors during a healthy run"

# Send a message during the loop
def send_test_messages():
    time.sleep(3)
    client.send_message("System", "Test message 1")
    time.sleep(5)
    client.send_message("System", "Test message 2")

sender = threading.Thread(target=send_test_messages)
sender.start()

robust_long_polling_loop(duration=15)

sender.join()

## 🧪 Quick Quiz

1. **Why does long polling have lower latency than simple polling?**

2. **What happens if a message arrives right after the client disconnects from a long poll?**

3. **You're building a payment status checker. Would you use simple polling or long polling?**

In [ ]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. Long polling responds IMMEDIATELY when data is available.")
print("   Simple polling waits for the next poll interval.")
print("")
print("2. The client will get it on the NEXT long poll request.")
print("   There's a brief window where messages can be delayed.")
print("   (This is why SSE/WebSocket can be better!)")
print("")
print("3. LONG POLLING! You want to know immediately when the")
print("   payment completes. The server can respond the instant")
print("   the payment status changes.")

## 📚 Summary

### What We Learned:

1. **Long polling** = Server holds request until data is available
2. **Lower latency** than simple polling for infrequent updates
3. **Still uses HTTP** - no special infrastructure needed
4. **Latency issue** with rapid consecutive messages
5. **Good for**: Payment status, job completion, infrequent updates

### Comparison Table:

| Feature | Simple Polling | Long Polling |
|---------|---------------|---------------|
| Latency | Up to poll interval | Near real-time |
| Server resources | Low (quick responses) | Higher (held connections) |
| Implementation | Very simple | Slightly complex |
| Burst messages | Consistent latency | Can add latency |

### Interview Tips:

> "Long polling is a good upgrade from simple polling when we need faster updates but don't want to add WebSocket complexity. It's perfect for infrequent but time-sensitive updates like payment confirmations."

### Next Up: Server-Sent Events (SSE)

In the next notebook, we'll see how **SSE** solves the burst message problem by keeping a single connection open for multiple updates!